In [4]:
import scipy
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib import cm

# --- Data Loading Functions ---
def load_mat_file(mat_path):
    data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    return data

def mat_struct_to_dict(obj):
    if isinstance(obj, np.ndarray):
        if obj.dtype == 'O':
            return [mat_struct_to_dict(o) for o in obj]
        else:
            return obj
    elif hasattr(obj, '_fieldnames'):
        return {field: mat_struct_to_dict(getattr(obj, field)) for field in obj._fieldnames}
    else:
        return obj

def get_session_dict(data_dict1):
    try:
        data_dict = mat_struct_to_dict(data_dict1['data'])
    except KeyError:
        data_dict = mat_struct_to_dict(data_dict1['ans'])
    return data_dict

def get_drilled_down(data_dict, session_key, subkey):
    return data_dict[session_key][subkey]


In [5]:
# Example usage:
mat_path = 'BG_031_250325.mat'

# Extract subject and session from mat_path
mat_base = os.path.splitext(os.path.basename(mat_path))[0]
parts = mat_base.split('_')
subject_id = '_'.join(parts[:2])  # e.g., 'BG_031'
session_id = mat_base             # e.g., 'BG_031_260325'

data = load_mat_file(mat_path)
data_dict1 = mat_struct_to_dict(data)
data_dict = get_session_dict(data_dict1)
session = get_drilled_down(data_dict, subject_id, session_id)



In [8]:
session_id

'BG_031_250325'

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

def collect_valid_pulses(session, fast_pulse_thresh=0.25, slow_pulse_thresh=-0.25, pre_window=[-0.4, 0], post_window=[0, 0.5]):
    """
    Collect valid fast and slow pulses that the subject actually saw, using event times.
    Returns: fast_pulse_times, slow_pulse_times
    """
    trials = session['behav_data']['trials_data_exp']
    ni_events = session['NI_events']
    all_fast_pulse_times = []
    all_slow_pulse_times = []
    for trial_idx, trial in enumerate(trials):
        TF_vec_full = np.array(trial['St1TrialVector'])
        TF_vec = TF_vec_full[::3]
        # Get Baseline_ON for this trial
        if 'Baseline_ON' in ni_events:
            baseline_on = ni_events['Baseline_ON']
            if isinstance(baseline_on, dict) and 'rise_t' in baseline_on:
                baseline_on_times = np.array(baseline_on['rise_t']).flatten()
            else:
                baseline_on_times = np.array(baseline_on).flatten()
            t0 = baseline_on_times[trial_idx]
        else:
            continue
        # Get Change_ON for this trial (if available)
        if 'Change_ON' in ni_events:
            change_on = ni_events['Change_ON']
            if isinstance(change_on, dict) and 'rise_t' in change_on:
                change_on_times = np.array(change_on['rise_t']).flatten()
            else:
                change_on_times = np.array(change_on).flatten()
            t_change = change_on_times[trial_idx] if trial_idx < len(change_on_times) else None
        else:
            t_change = None
        # Get outcome and outcome time for this trial
        outcome = trial.get('trialoutcome', None)
        reactiontimes = trial.get('reactiontimes', {})
        if outcome in ['FA', 'abort']:
            t_outcome = reactiontimes.get(outcome, np.nan)
            if not np.isnan(t_outcome):
                t_outcome = t0 + t_outcome
            else:
                t_outcome = None
        else:
            t_outcome = None

        log2_TF = np.log2(TF_vec)
        fast_pulse_bins = np.where(log2_TF >= fast_pulse_thresh)[0]
        slow_pulse_bins = np.where(log2_TF <= slow_pulse_thresh)[0]
        fast_pulse_times = (fast_pulse_bins * 0.05) + t0
        slow_pulse_times = (slow_pulse_bins * 0.05) + t0

        # Apply filtering conditions as in tf_responsive_unit_screening_v2
        def filter_pulse_times(pulse_times):
            valid = []
            for pt in pulse_times:
                if pt < t0 + 1.0:
                    continue
                if t_change is not None and pt > t_change - 1.0:
                    continue
                if outcome in ['FA', 'abort'] and t_outcome is not None and pt > t_outcome - 2.0:
                    continue
                valid.append(pt)
            return valid

        all_fast_pulse_times.extend(filter_pulse_times(fast_pulse_times))
        all_slow_pulse_times.extend(filter_pulse_times(slow_pulse_times))
    return np.array(all_fast_pulse_times), np.array(all_slow_pulse_times)

def mean_activity_per_neuron(session, pulse_times, pre_window, post_window, dt, sigma_ms):
    """
    For each cluster, compute the mean spike train across all pulses, then smooth the mean vector.
    Also compute the SEM across pulses (after smoothing each pulse's train).
    """
    npx_probes = session['NPX_probes']
    spike_times = npx_probes['st']
    cluster_ids = npx_probes['clu']
    good_clusters = npx_probes.get('cluster_id_KS_good', np.unique(cluster_ids))
    full_window = [pre_window[0], post_window[1]]
    t_vec = np.arange(full_window[0], full_window[1], dt)
    mean_activities = {}
    sem_activities = {}
    n_clusters = len(good_clusters)
    n_pulses = len(pulse_times)
    print(f"[INFO] Calculating mean activity for {n_clusters} clusters, {n_pulses} pulses")
    if n_pulses == 0:
        print("[WARN] No valid pulses found for this condition!")
    for i, clu in enumerate(good_clusters):
        if i % 10 == 0 or i == n_clusters - 1:
            print(f"[INFO] Processing cluster {i+1}/{n_clusters} (clu={clu})")
        spike_times_clu = spike_times[cluster_ids == clu]
        all_trains = []
        for t_pulse in pulse_times:
            aligned_spikes = spike_times_clu - t_pulse
            mask = (aligned_spikes >= full_window[0]) & (aligned_spikes < full_window[1])
            spikes_in_window = aligned_spikes[mask]
            spike_train = np.zeros_like(t_vec)
            spike_indices = np.searchsorted(t_vec, spikes_in_window)
            spike_indices = spike_indices[(spike_indices >= 0) & (spike_indices < len(spike_train))]
            spike_train[spike_indices] = 1
            all_trains.append(spike_train)
        if all_trains:
            all_trains = np.array(all_trains)
            sigma = sigma_ms / 1000 / dt
            # Smooth each pulse's train, then compute mean and SEM
            all_trains_smooth = np.array([gaussian_filter1d(train, sigma=sigma) for train in all_trains]) #, mode='constant', cval=0.0) 
            mean_activities[clu] = np.mean(all_trains_smooth, axis=0)
            sem_activities[clu] = np.std(all_trains_smooth, axis=0) / np.sqrt(all_trains_smooth.shape[0])
        else:
            mean_activities[clu] = np.zeros_like(t_vec)
            sem_activities[clu] = np.zeros_like(t_vec)
    print("[INFO] Finished mean activity calculation for all clusters.")
    return mean_activities, sem_activities, t_vec

def pre_pulse_stats(mean_activities, t_vec, pre_window):
    pre_mask = (t_vec >= pre_window[0]) & (t_vec < pre_window[1])
    stats = {}
    for clu, activity in mean_activities.items():
        pre_vals = activity[pre_mask]
        stats[clu] = {'mean': np.mean(pre_vals), 'std': np.std(pre_vals)}
    return stats

def zscore_activities(mean_activities, stats):
    z_activities = {}
    for clu, activity in mean_activities.items():
        mean = stats[clu]['mean']
        std = stats[clu]['std']
        if std > 0:
            z_activities[clu] = (activity - mean) / std
        else:
            z_activities[clu] = activity * 0
    return z_activities

def zscore_sems(sem_activities, stats):
    z_sems = {}
    for clu, sem in sem_activities.items():
        std = stats[clu]['std']
        if std > 0:
            z_sems[clu] = sem / std
        else:
            z_sems[clu] = sem * 0
    return z_sems


def plot_fast_slow_psth(z_fast, z_slow, t_vec, sem_fast=None, sem_slow=None, clusters=None, n_cols=10, trim_sigma=2, sigma_ms=13.3, dt=0.001, save_figure=True, session_id=None):
    """
    Plot mean PSTH for each cluster, with fast and slow pulse responses and error shading.
    Trims the edges to avoid smoothing artifacts.
    """
    if clusters is None:
        clusters = sorted(set(z_fast.keys()) & set(z_slow.keys()))
    n_plot = len(clusters)
    n_rows = int(np.ceil(n_plot / n_cols))
    # Calculate trim in points
    sigma = sigma_ms / 1000 / dt
    trim = int(np.ceil(trim_sigma * sigma))
    t_vec_trim = t_vec[trim:-trim] if trim > 0 else t_vec

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.2 * n_cols, 1.0 * n_rows), sharex=True, sharey=False)
    axes = axes.flatten()
    for idx, clu in enumerate(clusters):
        ax = axes[idx]
        # Trimmed data for plotting
        zf = z_fast[clu][trim:-trim] if trim > 0 else z_fast[clu]
        zs = z_slow[clu][trim:-trim] if trim > 0 else z_slow[clu]
        ax.plot(t_vec_trim, zf, color='r', label='Fast')
        ax.plot(t_vec_trim, zs, color='b', label='Slow')
        # Add error shading if SEMs are provided
        if sem_fast is not None:
            sf = sem_fast[clu][trim:-trim] if trim > 0 else sem_fast[clu]
            ax.fill_between(t_vec_trim, zf - sf, zf + sf, color='r', alpha=0.2)
        if sem_slow is not None:
            ss = sem_slow[clu][trim:-trim] if trim > 0 else sem_slow[clu]
            ax.fill_between(t_vec_trim, zs - ss, zs + ss, color='b', alpha=0.2)
        ax.axvline(0, color='k', linestyle='--', lw=0.8)
        # set y axis limits
        ax.set_ylim(-5, 5)
        ax.set_title(f'Clu {clu}', fontsize=9)
        # if idx % n_cols == 0:
        #     # ax.set_ylabel('Z-scored activity')
        # if idx // n_cols == n_rows - 1:
        #     ax.set_xlabel('Time from pulse (s)')
        if idx == 0:
            ax.legend(fontsize=8)
    for ax in axes[n_plot:]:
        ax.axis('off')
    plt.tight_layout(h_pad=0.2, w_pad=0.05)
    fig.text(0.007, 0.5, 'Z-scored activity', va='center', rotation='vertical', fontsize=12)
    fig.text(0.5, 0.007, 'Time from pulse (s)', ha='center', va='center', fontsize=12)
    if save_figure:
        output_dir = 'png_output'
        os.makedirs(output_dir, exist_ok=True)
        fig.savefig(os.path.join(output_dir, f'fast_slow_compare_session_{session_id}.png'), dpi=150)
        plt.close(fig)
    else:
        plt.show()


In [23]:
plot_fast_slow_psth(z_fast, z_slow, t_vec, sem_fast=sem_fast, sem_slow=sem_slow, clusters=list(z_fast.keys()), n_cols=10, trim_sigma=2, sigma_ms=13.3, dt=0.001, save_figure=True, session_id=session_id)

In [19]:
# Example usage: Run fast/slow pulse analysis and plot results
# Assumes you have a 'session' dictionary loaded as in your other scripts/notebooks

# --- Parameters ---
pre_window = [-0.4, 0]
post_window = [0, 0.5]
dt = 0.001
sigma_ms = 13.3

# --- Collect valid pulses ---
fast_pulse_times, slow_pulse_times = collect_valid_pulses(
    session,
    fast_pulse_thresh=0.25,
    slow_pulse_thresh=-0.25,  # updated to -0.25 for slow pulses
    pre_window=pre_window,
    post_window=post_window
)

# --- Mean activity for fast and slow pulses ---
mean_fast, sem_fast, t_vec = mean_activity_per_neuron(session, fast_pulse_times, pre_window, post_window, dt, sigma_ms)
mean_slow, sem_slow, _ = mean_activity_per_neuron(session, slow_pulse_times, pre_window, post_window, dt, sigma_ms)

# --- Pre-pulse stats ---
stats_fast = pre_pulse_stats(mean_fast, t_vec, pre_window)
stats_slow = pre_pulse_stats(mean_slow, t_vec, pre_window)

# --- Z-score ---
z_fast = zscore_activities(mean_fast, stats_fast)
z_slow = zscore_activities(mean_slow, stats_slow)

z_sem_fast = zscore_sems(sem_fast, stats_fast)
z_sem_slow = zscore_sems(sem_slow, stats_slow)


[INFO] Calculating mean activity for 218 clusters, 9250 pulses
[INFO] Processing cluster 1/218 (clu=2)
[INFO] Processing cluster 11/218 (clu=18)
[INFO] Processing cluster 21/218 (clu=40)
[INFO] Processing cluster 31/218 (clu=53)
[INFO] Processing cluster 41/218 (clu=72)
[INFO] Processing cluster 51/218 (clu=94)
[INFO] Processing cluster 61/218 (clu=118)
[INFO] Processing cluster 71/218 (clu=134)
[INFO] Processing cluster 81/218 (clu=162)
[INFO] Processing cluster 91/218 (clu=187)
[INFO] Processing cluster 101/218 (clu=200)
[INFO] Processing cluster 111/218 (clu=213)
[INFO] Processing cluster 121/218 (clu=233)
[INFO] Processing cluster 131/218 (clu=257)
[INFO] Processing cluster 141/218 (clu=280)
[INFO] Processing cluster 151/218 (clu=299)
[INFO] Processing cluster 161/218 (clu=311)
[INFO] Processing cluster 171/218 (clu=331)
[INFO] Processing cluster 181/218 (clu=378)
[INFO] Processing cluster 191/218 (clu=395)
[INFO] Processing cluster 201/218 (clu=409)
[INFO] Processing cluster 211/2

In [21]:

# --- Plot PSTHs ---
# plot_fast_slow_psth(z_fast, z_slow, t_vec)
# --- Plot PSTHs with z-scored SEMs ---
plot_fast_slow_psth(z_fast, z_slow, t_vec, sem_fast=z_sem_fast, sem_slow=z_sem_slow, save_figure=True, session_id=session_id)


In [22]:
session_id

'BG_031_250325'

## Save analysis results for quick reloading
Save all relevant results to a pickle file with an informative name (including session_id) after running the analysis.

In [20]:
import pickle

# Create results dictionary
results = {
    'fast_pulse_times': fast_pulse_times,
    'slow_pulse_times': slow_pulse_times,
    'mean_fast': mean_fast,
    'sem_fast': sem_fast,
    'mean_slow': mean_slow,
    'sem_slow': sem_slow,
    'z_fast': z_fast,
    'z_slow': z_slow,
    'z_sem_fast': z_sem_fast,
    'z_sem_slow': z_sem_slow,
    't_vec': t_vec,
    'pre_window': pre_window,
    'post_window': post_window,
    'dt': dt,
    'sigma_ms': sigma_ms,
    'session_id': session_id
}

# Save to pickle file
output_dir = 'pkls'
os.makedirs(output_dir, exist_ok=True)
pkl_path = os.path.join(output_dir, f'fast_slow_results_{session_id}.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(results, f)
print(f"Results saved to {pkl_path}")

Results saved to pkls\fast_slow_results_BG_031_250325.pkl


## Load previously saved analysis results
To skip recomputation, load the results from the pickle file for further analysis or plotting.

In [ ]:
import pickle
import os
session_id = 'BG_031_250325'
# Load results from pickle file
pkl_path = os.path.join('pkls', f'fast_slow_results_{session_id}.pkl')
with open(pkl_path, 'rb') as f:
    results = pickle.load(f)

# Unpack variables as needed
fast_pulse_times = results['fast_pulse_times']
slow_pulse_times = results['slow_pulse_times']
mean_fast = results['mean_fast']
sem_fast = results['sem_fast']
mean_slow = results['mean_slow']
sem_slow = results['sem_slow']
z_fast = results['z_fast']
z_slow = results['z_slow']
z_sem_fast = results['z_sem_fast']
z_sem_slow = results['z_sem_slow']
t_vec = results['t_vec']
pre_window = results['pre_window']
post_window = results['post_window']
dt = results['dt']
sigma_ms = results['sigma_ms']
session_id = results['session_id']

print(f"Results loaded from {pkl_path}")

In [ ]:
plot_fast_slow_psth(z_fast, z_slow, t_vec, sem_fast=z_sem_fast, sem_slow=z_sem_slow, save_figure=False, session_id=session_id)